## Código 1 — Descarga y preparación de datos PVWatts

en esta primera instancia se descarga una simulación horaria desde PVWatts para Rancagua.

PVWatts entrega directamente:

- `dni`: irradiancia directa normal.
- `dhi`: irradiancia difusa horizontal.
- `poa`: irradiancia sobre el plano del panel.
- `dc`: potencia DC simulada.
- `ac`: potencia AC simulada.
- `temperature`: temperatura ambiente.
- `tcell_pvwatts`: temperatura de celda estimada por PVWatts.
- `wind_speed`: velocidad del viento.
- `albedo`: reflectancia del suelo.

Luego, el código calcula:

- `ghi = dni · cos(zenith) + dhi`
- `fd = dhi / ghi`                    ----Fracción difusa
- `regimen_fd`, clasificando cada hora en:
  - `baja_fd`
  - `mixto`
  - `alta_fd`

Finalmente, guarda una base limpia en formato CSV para usarla en los siguientes pasos del análisis.

In [ ]:
import json
from pathlib import Path
from datetime import timezone, timedelta

import numpy as np
import pandas as pd
import requests
import pvlib


API_KEY = "rpKQ7B8DbjeA5ZIzu0254FiWbWf1USkJmFJpED2y"

# Ubicacion: Rancagua, Chile
LAT = -34.1708
LON = -70.7406

# Parametros estandar del sistema fotovoltaico
SYSTEM_CAPACITY = 1.0   # kW DC
AZIMUTH = 0             # norte, recomendado para hemisferio sur
TILT = 20               # grados
ARRAY_TYPE = 1          # fixed roof mounted
MODULE_TYPE = 0         # standard
LOSSES = 14             # %

BASE_YEAR = 2021        # solo para construir fechas; no es año historico real
GHI_MIN_FOR_FD = 10

OUT_DIR = Path("data")
OUT_FILE = OUT_DIR / "01_pvwatts_rancagua.csv"
OUT_META = OUT_DIR / "01_pvwatts_rancagua_metadata.json"


# DESCARGA PVWATTS
def request_pvwatts(dataset: str) -> dict:
    url = "https://developer.nlr.gov/api/pvwatts/v8.json"

    params = {
        "api_key": API_KEY,
        "lat": LAT,
        "lon": LON,
        "system_capacity": SYSTEM_CAPACITY,
        "azimuth": AZIMUTH,
        "tilt": TILT,
        "array_type": ARRAY_TYPE,
        "module_type": MODULE_TYPE,
        "losses": LOSSES,
        "timeframe": "hourly",
        "dataset": dataset,
    }

    response = requests.get(url, params=params, timeout=90)

    print(f"\nIntento con dataset='{dataset}'")
    print("Status code:", response.status_code)

    try:
        data = response.json()
    except ValueError:
        print("Respuesta no JSON:")
        print(response.text[:1000])
        response.raise_for_status()
        raise

    if response.status_code != 200:
        print("Respuesta de PVWatts:")
        print(json.dumps(data, indent=2, ensure_ascii=False)[:2000])
        response.raise_for_status()

    if data.get("errors"):
        print("Errores reportados por PVWatts:")
        print(data["errors"])
        raise RuntimeError(f"Errores desde PVWatts con dataset={dataset}: {data['errors']}")

    return data


def download_pvwatts():
    """
    Primero intenta con NSRDB.
    Si Rancagua no está disponible ahí para PVWatts, intenta con dataset internacional.
    """
    datasets_to_try = ["nsrdb", "intl"]

    last_error = None

    for dataset in datasets_to_try:
        try:
            data = request_pvwatts(dataset)
            data["_dataset_used"] = dataset
            return data

        except Exception as e:
            last_error = e
            print(f"Fallo con dataset='{dataset}': {e}")

    raise RuntimeError(
        "No se pudo descargar PVWatts para esta ubicacion con los datasets probados. "
        f"Ultimo error: {last_error}"
    )


# ARMAR DATAFRAME
def build_dataframe(data):
    outputs = data["outputs"]
    station_info = data["station_info"]

    tz_offset = float(station_info["tz"])
    tzinfo = timezone(timedelta(hours=tz_offset))

    n = len(outputs["dc"])

    datetime_local = pd.date_range(
        start=f"{BASE_YEAR}-01-01 00:00:00",
        periods=n,
        freq="h",
        tz=tzinfo,
    )

    df = pd.DataFrame({
        "datetime_local": datetime_local.tz_localize(None),
        "year": datetime_local.year,
        "month": datetime_local.month,
        "day": datetime_local.day,
        "hour": datetime_local.hour,

        # Variables solares PVWatts
        "dni": outputs["dn"],
        "dhi": outputs["df"],
        "poa": outputs["poa"],

        # Potencia simulada
        "dc": outputs["dc"],
        "ac": outputs["ac"],

        # Variables meteorologicas/simuladas
        "temperature": outputs["tamb"],
        "tcell_pvwatts": outputs["tcell"],
        "wind_speed": outputs["wspd"],
        "albedo": outputs["alb"],
    })

    # Calcular GHI aproximado desde DNI, DHI y posicion solar
    solar_position = pvlib.solarposition.get_solarposition(
        time=datetime_local,
        latitude=float(station_info["lat"]),
        longitude=float(station_info["lon"]),
    )

    cos_zenith = np.cos(np.deg2rad(solar_position["zenith"].to_numpy()))
    cos_zenith = np.where(cos_zenith > 0, cos_zenith, 0)

    df["ghi"] = df["dni"] * cos_zenith + df["dhi"]
    df["ghi"] = df["ghi"].clip(lower=0)

    # Fraccion difusa: fd = DHI / GHI
    df["fd"] = np.nan
    mask = df["ghi"] > GHI_MIN_FOR_FD
    df.loc[mask, "fd"] = df.loc[mask, "dhi"] / df.loc[mask, "ghi"]

    df.loc[(df["fd"] < 0) | (df["fd"] > 1), "fd"] = np.nan

    df["regimen_fd"] = pd.cut(
        df["fd"],
        bins=[-np.inf, 0.3, 0.7, np.inf],
        labels=["baja_fd", "mixto", "alta_fd"],
        right=False,
    )

    df = df[
        [
            "datetime_local",
            "year",
            "month",
            "day",
            "hour",
            "ghi",
            "dhi",
            "dni",
            "fd",
            "regimen_fd",
            "poa",
            "dc",
            "ac",
            "temperature",
            "tcell_pvwatts",
            "wind_speed",
            "albedo",
        ]
    ]

    return df


def save_metadata(data):
    metadata = {
        "note": "PVWatts entrega una simulacion horaria tipo TMY, no datos historicos reales.",
        "location_requested": {
            "lat": LAT,
            "lon": LON,
            "city": "Rancagua, Chile",
        },
        "system_parameters": {
            "system_capacity_kw": SYSTEM_CAPACITY,
            "azimuth": AZIMUTH,
            "tilt": TILT,
            "array_type": ARRAY_TYPE,
            "module_type": MODULE_TYPE,
            "losses_percent": LOSSES,
        },
        "dataset_used": data.get("_dataset_used"),
        "station_info": data.get("station_info"),
        "warnings": data.get("warnings"),
        "version": data.get("version"),
    }

    with open(OUT_META, "w", encoding="utf-8") as f:
        json.dump(metadata, f, indent=4, ensure_ascii=False)



# EJECUCION
def main():
    OUT_DIR.mkdir(parents=True, exist_ok=True)

    print("Descargando PVWatts para Rancagua...")
    data = download_pvwatts()

    print("\nDataset usado:")
    print(data.get("_dataset_used"))

    print("\nStation info:")
    print(json.dumps(data.get("station_info"), indent=2, ensure_ascii=False))

    print("\nArmando base...")
    df = build_dataframe(data)

    print("Guardando archivo...")
    df.to_csv(OUT_FILE, index=False)
    save_metadata(data)

    print("\nArchivo guardado:")
    print(OUT_FILE)

    print("\nMetadata guardada:")
    print(OUT_META)

    print("\nDimensiones:")
    print(df.shape)

    print("\nColumnas:")
    print(df.columns.tolist())

    print("\nConteo por regimen_fd:")
    print(df["regimen_fd"].value_counts(dropna=False))

    print("\nResumen:")
    print(
        df[
            [
                "ghi",
                "dhi",
                "dni",
                "fd",
                "poa",
                "dc",
                "ac",
                "temperature",
                "wind_speed",
            ]
        ].describe()
    )

    print("\nPrimeras filas:")
    print(df.head(24))


if __name__ == "__main__":
    main()

Descargando PVWatts para Rancagua...

Intento con dataset='nsrdb'
Status code: 422
Respuesta de PVWatts:
{
  "inputs": {
    "lat": "-34.1708",
    "lon": "-70.7406",
    "system_capacity": "1.0",
    "azimuth": "0",
    "tilt": "20",
    "array_type": "1",
    "module_type": "0",
    "losses": "14",
    "timeframe": "hourly",
    "dataset": "nsrdb"
  },
  "errors": [
    "No climate data found with dataset=nsrdb for location specified: lat=-34.1708 lon=-70.7406"
  ],
  "warnings": [
    "This location appears to be outside the US, try re-submitting with dataset=intl to check for international data"
  ],
  "version": "8.5.0",
  "ssc_info": {
    "version": 280,
    "build": "Linux 64 bit GNU/C++ Oct 18 2023 07:13:03",
    "module": "pvwattsv8"
  },
  "outputs": {}
}
Fallo con dataset='nsrdb': 422 Client Error: Unprocessable Entity for url: https://developer.nlr.gov/api/pvwatts/v8.json?api_key=rpKQ7B8DbjeA5ZIzu0254FiWbWf1USkJmFJpED2y&lat=-34.1708&lon=-70.7406&system_capacity=1.0&azimuth